# Lab Tracking trên Colab (GPU)

Chạy từ trên xuống. Chọn **Runtime → Change runtime type → T4 GPU** trước khi bắt đầu.

Notebook này làm thay các lệnh trong `HUONG_DAN.md`: cài đặt, tải dữ liệu, thử tracker, chạy bản nộp đủ frame, chấm `video_1`. Phần **bạn phải tự làm** là xem video và chọn cấu hình ở các ô có ghi `# CHỈNH`.

## 1. Cài đặt

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "KHÔNG có GPU - đổi runtime sang T4")

In [ ]:
# CHỈNH nếu repo hoặc nhánh của bạn khác.
REPO_BRANCH = "colab-lab"
REPO_URL = "https://github.com/Doan0904/K4-Track4-2A202602927-DangDinhDoan-Day22-Tracking-Student.git"

!git clone -q -b {REPO_BRANCH} {REPO_URL} /content/lab
%cd /content/lab
!pip install -q -r requirements.txt
!git clone -q https://github.com/JonathonLuiten/TrackEval.git /content/TrackEval
!pip install -q -e /content/TrackEval

Nếu `pip` báo xung đột `numpy` hoặc `import boxmot` lỗi: chọn **Runtime → Restart session**, rồi chạy tiếp từ ô tải dữ liệu (đừng cài lại).

## 2. Dữ liệu

In [ ]:
%cd /content/lab
!pip install -q gdown
!gdown --fuzzy "https://drive.google.com/file/d/1UeVPQd6j5pSzxoJDcKJrerT9SL3vJLDt/view?usp=sharing" -O /content/data_lab21.zip
!mkdir -p /content/unzipped && unzip -q -o /content/data_lab21.zip -d /content/unzipped

In [ ]:
import os
from pathlib import Path

# Tìm thư mục chứa video_1 ... video_5, dù zip có thêm một tầng thư mục.
hits = sorted(Path("/content/unzipped").rglob("video_1"))
assert hits, "Không thấy video_1 trong file zip"
LAB_DATA = str(hits[0].parent)
os.environ["LAB_DATA"] = LAB_DATA
print("LAB_DATA =", LAB_DATA)
!python scripts/check_data.py --lab-data-root "$LAB_DATA"
!python -m pytest -q

## 3. Hàm hỗ trợ

`show_previews` thu nhỏ video xem thử sang H.264 rồi nhúng vào notebook, kèm tên cấu hình.

In [ ]:
import base64
import subprocess
from IPython.display import HTML, display

DEVICE = "cuda:0"

def show_video(mp4, title, width=480):
    """Thu nhỏ video sang H.264 rồi nhúng vào notebook (Colab không phát được đường dẫn tương đối)."""
    mp4 = Path(mp4)
    web = mp4.with_name(mp4.stem + "_web.mp4")
    if not web.exists():
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(mp4),
                        "-vf", "scale=640:-2", "-vcodec", "libx264", "-crf", "32",
                        "-pix_fmt", "yuv420p", str(web)], check=True)
    data = base64.b64encode(web.read_bytes()).decode()
    display(HTML(f"<b>{title}</b><br><video width={width} controls loop "
                 f"src='data:video/mp4;base64,{data}'></video>"))

def show_previews(root, video):
    for mp4 in sorted(Path(root).glob(f"{video}_*/{video}_preview.mp4")):
        show_video(mp4, mp4.parent.name)

## 4. Baseline (bước D)

`video_1` + ByteTrack, 150 frame đầu.

In [ ]:
!python scripts/run_tracking.py --source "$LAB_DATA/video_1/img1" --seq-name video_1 \
  --tracker bytetrack --conf 0.3 --iou 0.5 --out runs/thu_nhanh --save-video --max-frames 150 --device {DEVICE}

## 5. `video_1`: chấm số (có nhãn)

**5a.** Cả 5 tracker ở cấu hình gốc, chạy đủ frame, có HOTA / MOTA / IDF1. Mất vài phút mỗi tracker trên T4.

In [ ]:
!python scripts/sweep_tracking.py --lab-data-root "$LAB_DATA" --video video_1 \
  --trackers bytetrack ocsort botsort strongsort deepocsort \
  --device {DEVICE} --trackeval-root /content/TrackEval --no-video --out runs/sweep

**5b.** Chọn tracker có HOTA cao nhất ở 5a, rồi quét `conf` 0.15 / 0.3 / 0.5 và `iou` 0.4 / 0.5 / 0.7, mỗi lần đổi một số.

In [ ]:
BEST_V1 = "bytetrack"  # CHỈNH: tracker tốt nhất ở bảng 5a

!python scripts/sweep_tracking.py --lab-data-root "$LAB_DATA" --video video_1 \
  --trackers {BEST_V1} --sweep-trackers {BEST_V1} \
  --device {DEVICE} --trackeval-root /content/TrackEval --no-video --out runs/sweep

## 6. `video_2` … `video_5`: xem bằng mắt

Với mỗi video: chạy hai nhóm tracker (chuyển động và Re-ID) ở cấu hình gốc, 150 frame đầu, có video. Xem và ghi: ID có đổi màu giữa chừng không, hộp có nhảy sang người bên cạnh không, có người rõ mà thiếu hộp không, có hộp nhấp nháy trên nền không.

In [ ]:
VIDEO = "video_2"  # CHỈNH: lần lượt video_2, video_3, video_4, video_5

!python scripts/sweep_tracking.py --lab-data-root "$LAB_DATA" --video {VIDEO} \
  --trackers bytetrack ocsort botsort strongsort deepocsort \
  --device {DEVICE} --max-frames 150 --out runs/eye
show_previews("runs/eye", VIDEO)

Chọn tracker trông tốt nhất cho video đó, rồi quét `conf` / `iou` của riêng nó (vẫn 150 frame).

In [ ]:
PICK = "botsort"  # CHỈNH: tracker bạn chọn cho VIDEO ở trên

!python scripts/sweep_tracking.py --lab-data-root "$LAB_DATA" --video {VIDEO} \
  --trackers {PICK} --sweep-trackers {PICK} \
  --device {DEVICE} --max-frames 150 --out runs/eye_sweep
show_previews("runs/eye_sweep", VIDEO)

## 7. Bản nộp: chạy đủ frame

Điền cấu hình cuối cho từng video theo điều bạn đã thấy ở bước 5 và 6. Giá trị dưới đây chỉ là điểm xuất phát, chưa phải kết quả đã kiểm chứng.

In [ ]:
CONFIG = {  # CHỈNH: (tracker, conf, iou)
    "video_1": ("bytetrack", 0.3, 0.5),
    "video_2": ("botsort", 0.3, 0.5),
    "video_3": ("botsort", 0.3, 0.5),
    "video_4": ("botsort", 0.3, 0.5),
    "video_5": ("botsort", 0.3, 0.5),
}

for video, (tracker, conf, iou) in CONFIG.items():
    print(f"\n##### {video}: {tracker} conf={conf} iou={iou}")
    subprocess.run([
        "python", "scripts/run_tracking.py",
        "--source", f"{LAB_DATA}/{video}/img1", "--seq-name", video,
        "--tracker", tracker, "--conf", str(conf), "--iou", str(iou),
        "--out", "runs/nop_bai", "--save-video", "--device", DEVICE,
    ], check=True)

Kiểm tra đủ frame: số frame cuối trong file kết quả phải bằng số ảnh trong `img1`.

In [ ]:
for video in CONFIG:
    n_imgs = len(list((Path(LAB_DATA) / video / "img1").glob("*.jpg")))
    rows = Path(f"runs/nop_bai/{video}.txt").read_text().splitlines()
    last = max(int(r.split(",")[0]) for r in rows) if rows else 0
    print(f"{video}: {n_imgs} ảnh, {len(rows)} dòng, frame cuối có hộp = {last}")

## 8. Chấm `video_1` cho báo cáo

In [ ]:
!python scripts/evaluate_practice.py --trackeval-root /content/TrackEval \
  --lab-data-root "$LAB_DATA" --submission runs/nop_bai/video_1.txt --run-name nop_bai_video1

Xem lại video bản nộp của cả năm video để viết quan sát cho báo cáo.

In [ ]:
for video in CONFIG:
    show_video(f"runs/nop_bai/{video}_preview.mp4", video)

## 9. Tải kết quả về

In [ ]:
!cd runs && zip -q -r nop_bai.zip nop_bai -x "*_web.mp4"
from google.colab import files
files.download("runs/nop_bai.zip")